# Проект 1. Почему в апреле 2026 упала выручка от подписок

**Автор:** Рубан Ирина, БАЦРФ262 · курс «Python для бизнес-аналитики», ВШБ НИУ ВШЭ

---

## Бизнес-вопрос

С 1 марта 2026 ПРАЙМ поднял цены: Базовый 199 → 249 ₽, Расширенный 399 → 449 ₽.
В марте выручка от подписок выросла до 58,2 млн ₽, а в апреле упала до 53,8 млн ₽,
то есть на **−4,4 млн ₽ (−7,6 %)**.

Вопрос руководителя: **«Повышение цен начало отпугивать клиентов? Откатывать цены или нет?»**

Вопрос проверяемый: если виновато повышение, в апреле должна сократиться база
(больше отмен с причиной «дорого») или клиенты должны уйти на дешёвый тариф.

## Гипотезы

| № | Гипотеза | Как проверяю |
|---|---|---|
| H1 | После повышения цен вырос отток с причиной `too_expensive` и база стала расти медленнее | отток «дорого» на 1000 активных подписок в день, до и после 1 марта |
| H2 | Апрель просто короче марта (30 дней против 31) | выручка в пересчёте на день |
| H3 | Часть апрельских платежей не прошла или не попала в базу | платежи по дням, фактор «платежей на подписку в день» в разложении |

Спойлер: **H1 не подтвердилась**, H2 объясняет меньшую часть падения, H3 — большую.

## 0. Подключение

Функция `q()` — как на семинаре 6: выполняет запрос и сразу закрывает соединение.
Внутри `set time zone 'UTC'`: месяцы и сутки считаем по UTC, тогда границы месяцев
у всех одинаковые.

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()

ACCENT = "#C1272D"   # один акцентный цвет — то, о чём вывод
GRAY = "#9CA3AF"     # всё остальное — серым


def q(sql: str, **params) -> pd.DataFrame:
    # Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу.
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def rub(x: float) -> str:
    # 12345678.9 -> '12 345 679'
    return f"{x:,.0f}".replace(",", " ")


print("подключились как:", q("select current_user as login").iloc[0, 0])

## 1. Зерно, период и сегменты

**Зерно: одна строка = месяц × тариф × канал привлечения владельца подписки.**

* **Месяц** — потому что вопрос про месячную выручку (март против апреля), а цена
  и списания устроены помесячно: подписка платит примерно раз в 30 дней.
* **Тариф** — потому что повышение цены было разным: +25 % у Базового и +12,5 %
  у Расширенного. Если клиенты «сбегают» на дешёвый тариф, это видно только в разрезе тарифа.
* **Канал** — чтобы проверить, не виноват ли один сегмент: если падение сидит в одном
  канале, причина в нём, если везде одинаково — причина общая (цена или техника).
* Мельче (платёж, день) не беру для разложения: на уровне платежа нет «активных подписок»,
  а на уровне дня выручка шумит. Дни понадобятся только для проверки H3.

Канал берём у **владельца** подписки (`subscriptions.owner_client_id`): он и платит.
Клиенты без канала — отдельной строкой «без канала», не выбрасываем: иначе части
не сойдутся с итогом.

**Подписка активна в месяце**, если началась до конца месяца и не закончилась до его начала
(определение с Л5/С6). Выручка — только платежи со статусом `success`.

**Период загрузки** — вся история, январь 2025 — август 2026, чтобы видеть тренд и сезонность.
Для выводов использую февраль 2025 — июль 2026: январь 2025 неполный (история начинается
6 января), август 2026 тоже неполный (история кончается 30 августа) и в нём есть
отдельная аномалия — о ней в ограничениях.

In [ ]:
# Месяц × тариф × канал: активные подписки, число успешных платежей, выручка.
# Две части считаем отдельно и соединяем по ключу (месяц, тариф, канал):
#   act — активные подписки (из subscriptions),
#   rev — платежи (из payments).
# subscription_members НЕ присоединяем: платит только владелец, участники
# размножили бы строки платежей (проверка ниже).
monthly = q('''
    with act as (
        select to_char(m.month_start, 'YYYY-MM') as month,
               s.tariff_id,
               c.acquisition_channel as channel,
               count(*) as active
        from generate_series(cast(:first as timestamptz), cast(:last as timestamptz),
                             interval '1 month') as m(month_start)
        join prime.subscriptions s
          on s.started_at < m.month_start + interval '1 month'
         and (s.ended_at is null or s.ended_at >= m.month_start)
        join prime.clients c on c.client_id = s.owner_client_id
        group by 1, 2, 3
    ),
    rev as (
        select to_char(p.paid_at, 'YYYY-MM') as month,
               s.tariff_id,
               c.acquisition_channel as channel,
               count(*) as n_payments,
               sum(p.amount) as revenue
        from prime.payments p
        join prime.subscriptions s on s.subscription_id = p.subscription_id
        join prime.clients c on c.client_id = s.owner_client_id
        where p.status = 'success'
          and p.paid_at >= cast(:first as timestamptz)
          and p.paid_at <  cast(:last as timestamptz) + interval '1 month'
        group by 1, 2, 3
    )
    select a.month, a.tariff_id, a.channel, a.active,
           coalesce(r.n_payments, 0) as n_payments,
           coalesce(r.revenue, 0)    as revenue
    from act a
    left join rev r
      on r.month = a.month
     and r.tariff_id = a.tariff_id
     and r.channel is not distinct from a.channel   -- NULL = NULL для «без канала»
    order by 1, 2, 3
''', first="2025-01-01", last="2026-08-01")

# Пропуск канала -> отдельная строка «без канала» (договорённость курса).
monthly["channel"] = monthly["channel"].fillna("без канала")
monthly["tariff"] = monthly["tariff_id"].map({1: "Базовый", 2: "Расширенный"})

# Зерно должно быть уникальным: одна строка на месяц × тариф × канал.
assert not monthly.duplicated(["month", "tariff_id", "channel"]).any(), "зерно не уникально"
assert monthly["channel"].isna().sum() == 0

print(len(monthly), "строк =", monthly["month"].nunique(), "месяцев ×",
      monthly["tariff_id"].nunique(), "тарифа ×", monthly["channel"].nunique(), "каналов")
monthly.head(7)

## 2. Проверки перед расчётом

Три ловушки из описания базы: двойной счёт через `subscription_members`, справочник
тарифов с историей и потерянные строки. Проверяю до того, как делать выводы.

In [ ]:
# 2.1. Сверка: выручка в витрине = выручка прямо из payments (без соединений).
direct = q('''
    select sum(amount) as revenue, count(*) as n_payments
    from prime.payments
    where status = 'success'
      and paid_at >= '2025-01-01' and paid_at < '2026-09-01'
''').iloc[0]

print("витрина:      ", rub(monthly["revenue"].sum()), "₽,", rub(monthly["n_payments"].sum()), "платежей")
print("прямой запрос:", rub(direct["revenue"]), "₽,", rub(direct["n_payments"]), "платежей")
assert abs(monthly["revenue"].sum() - float(direct["revenue"])) < 1, "витрина потеряла или задвоила выручку"
assert monthly["n_payments"].sum() == direct["n_payments"]

In [ ]:
# 2.2. Двойной счёт: что было бы, если присоединить subscription_members к платежам.
# Март–апрель 2026, только успешные платежи.
dbl = q('''
    select
      (select count(*) from prime.payments p
        where p.status = 'success' and p.paid_at >= '2026-03-01' and p.paid_at < '2026-05-01') as payments,
      (select count(*) from prime.payments p
        join prime.subscription_members sm on sm.subscription_id = p.subscription_id
        where p.status = 'success' and p.paid_at >= '2026-03-01' and p.paid_at < '2026-05-01') as with_members
''').iloc[0]

print("платежей:                    ", rub(dbl["payments"]))
print("строк после join с участниками:", rub(dbl["with_members"]),
      f"(в {dbl['with_members'] / dbl['payments']:.2f} раза больше)")
# Строк стало больше: каждый платёж повторился столько раз, сколько у подписки участников,
# а платежи подписок без участников вообще пропали бы. Поэтому в витрине этого join нет.

In [ ]:
# 2.3. Справочник тарифов: у одного tariff_id несколько строк с интервалами цен.
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")
print(tariffs[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]].to_string(index=False))

# У Расширенного интервалы пересекаются: строка 429 ₽ (02.2026–06.2026) накрывает
# и 399 ₽, и 449 ₽. Какая цена реально списывалась? Смотрим на сами платежи.
amounts = q('''
    select to_char(p.paid_at, 'YYYY-MM') as month, s.tariff_id, p.amount, count(*) as n
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success' and p.paid_at >= '2026-01-01' and p.paid_at < '2026-08-01'
    group by 1, 2, 3 order by 1, 2, 3
''')
print()
# pivot упадёт, если в месяце у тарифа больше одной суммы — это тоже проверка.
print(amounts.pivot(index="month", columns="tariff_id", values="amount"))
# 429 ₽ не списывалось ни разу: строка в справочнике ошибочная. Если бы я брал цену
# из справочника по дате, платежи Расширенного за март–июнь задвоились бы.
# Поэтому цену беру из самих платежей: price = revenue / n_payments.

In [ ]:
# 2.4. Нет ли дней совсем без платежей? (сутки по UTC)
gaps = q('''
    with d as (select generate_series(date '2025-01-06', date '2026-08-30', interval '1 day')::date as day),
         p as (select paid_at::date as day, count(*) as n from prime.payments group by 1)
    select d.day, coalesce(p.n, 0) as n_payments
    from d left join p on p.day = d.day
    where coalesce(p.n, 0) < 3000          -- в обычный день 5–6 тысяч платежей
    order by 1
''')
gaps
# 06.01.2025 и 30.08.2026 — края истории, неполные сутки.
# 14 и 15 апреля 2026 — НОЛЬ платежей, даже неуспешных. Это к гипотезе H3.

## 3. Что случилось: выручка по месяцам и в пересчёте на день

In [ ]:
m = monthly.groupby("month")[["active", "n_payments", "revenue"]].sum()
m["days"] = pd.PeriodIndex(m.index, freq="M").days_in_month
m["revenue_per_day"] = m["revenue"] / m["days"]
m["growth_pct"] = m["revenue"].pct_change() * 100
m["growth_per_day_pct"] = m["revenue_per_day"].pct_change() * 100

m.loc["2025-12":"2026-08", ["active", "n_payments", "revenue", "days",
                            "revenue_per_day", "growth_pct", "growth_per_day_pct"]].round(1)

In [ ]:
A, B = "2026-03", "2026-04"   # сравниваем март и апрель 2026
total_delta = m.loc[B, "revenue"] - m.loc[A, "revenue"]
print(f"выручка: {rub(m.loc[A, 'revenue'])} -> {rub(m.loc[B, 'revenue'])} ₽, "
      f"изменение {rub(total_delta)} ₽ ({m.loc[B, 'growth_pct']:+.1f} %)")
print(f"в день:  {rub(m.loc[A, 'revenue_per_day'])} -> {rub(m.loc[B, 'revenue_per_day'])} ₽, "
      f"{m.loc[B, 'growth_per_day_pct']:+.1f} %")
# Даже в пересчёте на день апрель ниже марта. Значит, длиной месяца (H2) всё не объяснить.

In [ ]:
# График 1. Выручка в день по месяцам. Период выводов: 02.2025–07.2026.
plot = m.loc["2025-02":"2026-07"]
colors = [ACCENT if month == B else GRAY for month in plot.index]

fig, ax = plt.subplots(figsize=(10, 4.4), layout="constrained")
ax.bar(plot.index, plot["revenue_per_day"] / 1e6, color=colors, zorder=2)
ax.axvline(list(plot.index).index("2026-03") - 0.5, color="#111827", linestyle="--", linewidth=0.8)
ax.text(list(plot.index).index("2026-03") - 0.4, 2.05, "повышение цен 01.03", fontsize=9)
ax.set_title("Апрель 2026 — единственный месяц, когда выручка в день упала", loc="left")
ax.set_ylabel("выручка в день, млн руб.")
ax.set_xlabel("месяц (UTC); выручка месяца / число дней в месяце")
ax.set_ylim(0, 2.3)                       # ось от нуля
ax.tick_params(axis="x", rotation=60)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB", zorder=0)
plt.show()

## 4. Где: аддитивная модель по каналам и тарифам

Выручка = сумма выручек каналов, значит её изменение = сумма изменений каналов.
Если сумма частей не равна итогу — где-то потерялись строки (как «без канала» на С6).

In [ ]:
by_channel = monthly.pivot_table(index="channel", columns="month", values="revenue", aggfunc="sum")
by_channel = by_channel[[A, B]].copy()
by_channel["delta"] = by_channel[B] - by_channel[A]
# Изменение выручки в день: убирает разницу 31 и 30 дней.
by_channel["per_day_pct"] = ((by_channel[B] / 30) / (by_channel[A] / 31) - 1) * 100
by_channel = by_channel.sort_values("delta")

# Сумма частей = итог (до копейки).
assert abs(by_channel["delta"].sum() - total_delta) < 0.01, "каналы не сходятся с итогом"

summary = by_channel.copy()
summary.loc["итого"] = [summary[A].sum(), summary[B].sum(), summary["delta"].sum(),
                        ((summary[B].sum() / 30) / (summary[A].sum() / 31) - 1) * 100]
summary.round({A: 0, B: 0, "delta": 0, "per_day_pct": 1})

In [ ]:
by_tariff = monthly.pivot_table(index="tariff", columns="month", values="revenue", aggfunc="sum")[[A, B]]
by_tariff["delta"] = by_tariff[B] - by_tariff[A]
by_tariff["per_day_pct"] = ((by_tariff[B] / 30) / (by_tariff[A] / 31) - 1) * 100
assert abs(by_tariff["delta"].sum() - total_delta) < 0.01, "тарифы не сходятся с итогом"

# Доля Расширенного в платежах: если бы клиенты бежали на дешёвый тариф, она бы падала.
pay_t = monthly.pivot_table(index="month", columns="tariff", values="n_payments", aggfunc="sum")
share_ext = (pay_t["Расширенный"] / pay_t.sum(axis=1) * 100).loc["2025-12":"2026-07"].round(1)

print(by_tariff.round({A: 0, B: 0, "delta": 0, "per_day_pct": 1}))
print()
print("доля Расширенного в платежах, %:")
print(share_ext.to_string())

**Вывод аддитивной модели.** Падение есть во **всех** семи строках каналов (включая «без канала»)
и в обоих тарифах, и в пересчёте на день оно почти одинаковое — около −4…−5 %.
Доля Расширенного в платежах не изменилась (~38 %): на дешёвый тариф никто не побежал.
Значит, причина не в одном сегменте, а общая — то, что задело всех сразу.

In [ ]:
# График 2. Изменение выручки в день по каналам: одинаковое везде.
plot = by_channel.sort_values("per_day_pct")
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
ax.barh(plot.index, plot["per_day_pct"], color=GRAY, zorder=2)
total_pct = summary.loc["итого", "per_day_pct"]
ax.axvline(total_pct, color=ACCENT, linewidth=2, label=f"все каналы: {total_pct:.1f} %")
ax.axvline(0, color="#111827", linewidth=0.8)
ax.set_title("Выручка в день упала во всех каналах почти одинаково — дело не в сегменте", loc="left")
ax.set_xlabel("изменение выручки в день, апрель 2026 к марту 2026, %")
ax.set_ylabel("канал привлечения владельца")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#E5E7EB", zorder=0)
plt.show()

## 5. За счёт чего: мультипликативная модель

Выручка раскладывается на множители:

> **выручка = активные подписки × платежей на подписку × цена платежа**

Второй множитель смешивает две вещи: сколько дней в месяце и как часто подписка
платит в один день. Разделяю их, чтобы отдельно увидеть гипотезы H2 и H3:

> **выручка = активные × дней в месяце × платежей на подписку в день × цена платежа**

Произведение четырёх множителей равно выручке ровно (дни сокращаются).

Порядок цепных подстановок — сначала количественные факторы, потом качественные:
активные → дни → платежей в день → цена. Сначала проверю функции на примерах с лекции.

In [ ]:
def chain(n0, p0, n1, p1):
    # Цепные подстановки для R = N × P: сначала количество (при старой цене), потом цена.
    by_count = (n1 - n0) * p0
    by_price = n1 * (p1 - p0)
    assert abs(by_count + by_price - (n1 * p1 - n0 * p0)) < 0.01, "сумма влияний не равна изменению"
    return round(by_count), round(by_price)


def contrib(x0: dict, x1: dict, order: list[str]) -> dict:
    # Влияние каждого фактора: меняем факторы по одному в порядке order,
    # остальные держим на текущем уровне.
    cur, out = dict(x0), {}
    for f in order:
        before = math.prod(cur.values())
        cur[f] = x1[f]                                   # фактор переходит на новое значение
        out[f] = math.prod(cur.values()) - before
    total = math.prod(x1.values()) - math.prod(x0.values())
    assert abs(sum(out.values()) - total) < 0.01, "сумма влияний не равна изменению"
    return out


# Пекарня с лекции: 100 булок по 50 ₽ -> 90 булок по 60 ₽. Ждём (-500, 900).
print(chain(100, 50, 90, 60))
# Игрушка с лекции: ждём 3000, -3300, 1980.
toy = contrib({"ops": 1000, "chk": 2000, "rate": 0.015},
              {"ops": 1100, "chk": 1800, "rate": 0.016}, ["ops", "chk", "rate"])
print({f: round(v) for f, v in toy.items()})

In [ ]:
FACTORS = ["active", "days", "per_sub_day", "price"]
LABELS = {"active": "активные подписки", "days": "дней в месяце",
          "per_sub_day": "платежей на подписку в день", "price": "цена платежа"}

m["per_sub_day"] = m["n_payments"] / m["active"] / m["days"]
m["price"] = m["revenue"] / m["n_payments"]

# Произведение факторов должно давать выручку каждого месяца.
check = m[FACTORS].prod(axis=1)
assert (check - m["revenue"]).abs().max() < 0.01, "факторы не перемножаются в выручку"

m.loc["2025-12":"2026-07", FACTORS].round(5)
# платежей на подписку в день держится около 0,0314 (раз в ~30 дней)
# и только в апреле 2026 проседает до 0,0294.

In [ ]:
x0 = m.loc[A, FACTORS].to_dict()
x1 = m.loc[B, FACTORS].to_dict()

effects = pd.Series(contrib(x0, x1, FACTORS))
assert abs(effects.sum() - total_delta) < 0.01, "сумма влияний не равна изменению выручки"

print(effects.rename(LABELS).map(rub).to_string())
print(f"\nсумма влияний {rub(effects.sum())} ₽ = изменение выручки {rub(total_delta)} ₽")

# Для сравнения — двухфакторная модель с С6: база × выручка на подписку.
by_active, by_rev_per_sub = chain(m.loc[A, "active"], m.loc[A, "revenue"] / m.loc[A, "active"],
                                  m.loc[B, "active"], m.loc[B, "revenue"] / m.loc[B, "active"])
print(f"\nдва фактора: база {rub(by_active)} ₽, выручка на подписку {rub(by_rev_per_sub)} ₽")

**Что видно.** База выросла и **добавила** выручку — то есть массового ухода клиентов нет.
Цена в марте и апреле одинаковая (249/449 ₽), её влияние около нуля.
Минус дают два фактора: короче месяц (H2) и — больше всего — **меньше платежей на подписку
в день** (H3). Подписка платит раз в 30 дней, частота сама по себе не меняется — значит,
в апреле часть платежей «пропала».

In [ ]:
# Сколько платежей должно было быть 14–15 апреля, если бы дни были обычные?
daily = q('''
    select paid_at::date as day, count(*) as n_payments, sum(amount) as revenue
    from prime.payments
    where status = 'success' and paid_at >= '2026-03-01' and paid_at < '2026-06-01'
    group by 1 order by 1
''')
daily["day"] = pd.to_datetime(daily["day"])
daily = daily.set_index("day").asfreq("D", fill_value=0)   # пустые дни -> явные нули

april = daily.loc["2026-04"]
normal_days = april[april["n_payments"] > 0]
expected_pay = normal_days["n_payments"].mean() * 2          # два пустых дня
expected_rev = expected_pay * m.loc[B, "price"]

print(f"обычный апрельский день: {rub(normal_days['n_payments'].mean())} платежей")
print(f"ожидалось 14–15 апреля:  ~{rub(expected_pay)} платежей ≈ {rub(expected_rev)} ₽")
print(f"влияние фактора «платежей в день» в разложении: {rub(effects['per_sub_day'])} ₽")

# Если вернуть эти два дня, выручка в день в апреле была бы выше мартовской:
fixed = (m.loc[B, "revenue"] + expected_rev) / 30
print(f"\nапрель с двумя днями: {rub(fixed)} ₽ в день против {rub(m.loc[A, 'revenue_per_day'])} в марте "
      f"({(fixed / m.loc[A, 'revenue_per_day'] - 1) * 100:+.1f} %)")

In [ ]:
# График 3. Платежи по дням, март–май 2026.
colors = [ACCENT if n == 0 else GRAY for n in daily["n_payments"]]
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.bar(daily.index, daily["n_payments"] / 1000, color=colors, width=0.8, zorder=2)
for d in ["2026-04-14", "2026-04-15"]:
    ax.axvspan(pd.Timestamp(d) - pd.Timedelta(hours=12), pd.Timestamp(d) + pd.Timedelta(hours=12),
               color=ACCENT, alpha=0.25, zorder=1)
ax.annotate("14–15 апреля: 0 платежей", xy=(pd.Timestamp("2026-04-15"), 0.3),
            xytext=(pd.Timestamp("2026-04-20"), 2.2), color=ACCENT,
            arrowprops={"arrowstyle": "->", "color": ACCENT})
ax.set_title("14 и 15 апреля 2026 не прошло ни одного платежа — обычно их ~5,9 тыс. в день", loc="left")
ax.set_ylabel("успешных платежей, тыс. в день")
ax.set_xlabel("день (UTC), март–май 2026")
ax.set_ylim(0, 7)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB", zorder=0)
plt.show()

In [ ]:
def waterfall(deltas: pd.Series, title: str, accent: str) -> plt.Axes:
    # Каскад изменений: столбики факторов «плывут» (каждый начинается там,
    # где закончился предыдущий), последний — «итого» от нуля.
    before = deltas.cumsum() - deltas
    colors = [ACCENT if name == accent else GRAY for name in deltas.index]

    fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
    bars = ax.bar(deltas.index, deltas, bottom=before, color=colors, zorder=2)
    total = ax.bar("итого", deltas.sum(), color="#4B5563", zorder=2)
    for group in (bars, total):
        ax.bar_label(group, fmt=lambda v: f"{v:+,.0f}".replace(",", " ").replace("-", "−"), padding=3)
    ax.axhline(0, color="#111827", linewidth=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("тыс. руб.")
    ax.use_sticky_edges = False
    ax.margins(y=0.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#E5E7EB", zorder=0)
    return ax


# График 4. Каскад разложения март -> апрель.
# Подписи переносим на две строки, чтобы не налезали друг на друга.
steps = effects.rename({f: LABELS[f].replace(" ", "\n", 1) for f in FACTORS}) / 1000
ax = waterfall(steps, "Выручку уронили пропавшие платежи, а не уход клиентов: база добавила +1,2 млн",
               accent=LABELS["per_sub_day"].replace(" ", "\n", 1))
ax.set_xlabel("Март → апрель 2026, тыс. руб. Порядок подстановок — слева направо.")
plt.show()

## 6. Гипотеза H1: повышение цен увеличило отток «дорого»

Если клиентов отпугнула цена, после 1 марта должна вырасти доля отмен с причиной
`too_expensive`. Считаю **на 1000 активных подписок в день**: база растёт,
а месяцы разной длины — без нормировки рост числа отмен ничего не значит.
Период: сентябрь 2025 — июль 2026, по полгода до и почти полгода после повышения
(август исключён — см. ограничения).

In [ ]:
churn = q('''
    select to_char(ended_at, 'YYYY-MM') as month,
           count(*) filter (where cancel_reason = 'too_expensive') as too_expensive,
           count(*) as ended
    from prime.subscriptions
    where ended_at >= '2025-09-01' and ended_at < '2026-08-01'
    group by 1 order by 1
''').set_index("month")

churn = churn.join(m[["active", "days"]])
churn["expensive_per_1000_day"] = churn["too_expensive"] / churn["active"] / churn["days"] * 1000
churn["all_per_1000_day"] = churn["ended"] / churn["active"] / churn["days"] * 1000
churn["after"] = churn.index >= "2026-03"

print(churn.round(3))
print()
print(churn.groupby("after")[["expensive_per_1000_day", "all_per_1000_day"]].mean()
      .rename(index={False: "до 03.2026", True: "с 03.2026"}).round(3))

In [ ]:
# График 5. Отток «дорого» до и после повышения цен.
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.plot(churn.index, churn["expensive_per_1000_day"], color=GRAY, marker="o", zorder=2)
ax.plot(churn.index[churn["after"]], churn.loc[churn["after"], "expensive_per_1000_day"],
        color=ACCENT, marker="o", linewidth=2, zorder=3, label="после повышения цен")
ax.axvline(list(churn.index).index("2026-03") - 0.5, color="#111827", linestyle="--", linewidth=0.8)
ax.set_title("После повышения цен отмен «слишком дорого» не стало больше — H1 не подтвердилась", loc="left")
ax.set_ylabel("отмен too_expensive\nна 1000 активных подписок в день")
ax.set_xlabel("месяц отмены (UTC)")
ax.set_ylim(0, 0.4)                      # ось от нуля
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB", zorder=0)
plt.show()

## 7. Вывод и рекомендация

**Что случилось.** Выручка от подписок в апреле 2026 — 53,8 млн ₽ против 58,2 млн ₽ в марте:
−4,4 млн ₽ (−7,6 %). В пересчёте на день — −4,6 %. Это единственный месяц за полтора года,
когда выручка в день упала.

**За счёт чего** (цепные подстановки, март → апрель):

* активные подписки: **+1,2 млн ₽** — база продолжила расти;
* дней в месяце (30 против 31): **−1,9 млн ₽** — эффект календаря (H2);
* платежей на подписку в день: **−3,8 млн ₽** — 14 и 15 апреля в базе нет ни одного
  платежа, ни успешного, ни неуспешного (H3). Ожидалось ~11,8 тыс. платежей ≈ 3,8 млн ₽;
* цена платежа: +0,02 млн ₽, то есть ≈ 0 — цены в марте и апреле одинаковые.

Падение одинаковое во всех каналах и обоих тарифах, доля Расширенного не изменилась.

**Гипотеза H1 не подтвердилась.** Отток с причиной «дорого» после повышения цен остался
на уровне ~0,3 отмены на 1000 активных подписок в день, общий отток тоже не вырос.
Без двух пустых дней апрельская выручка в день была бы на ~2,3 % выше мартовской — обычный темп роста.

**Рекомендация.**

1. **Цены не откатывать.** Повышение дало +18 % к цене платежа и не увеличило отток.
2. **Платёжной команде и команде данных — до 06.11.2026** сверить 14–15 апреля 2026
   с выпиской эквайринга: если списаний не было — это ~11,8 тыс. подписок и ~3,8 млн ₽
   недополученной выручки (повторно списать или проверить, что они заплатили позже);
   если списания были, но не попали в базу — дозагрузить, иначе отчётность занижена.
3. **Аналитике** — добавить в ежедневный мониторинг проверку «день без платежей»
   (как проверка 2.4 выше), чтобы такой сбой ловился в тот же день, а не через месяц.

**Ограничения.**

* По данным нельзя отличить «платежи не прошли» от «платежи не загрузились»: нужна сверка
  с биллингом.
* Оценка 3,8 млн ₽ — по среднему апрельскому дню; реальный график списаний мог отличаться.
* Причина отмены `cancel_reason` — то, что выбрал клиент; часть «дорогих» могла выбрать «другое».
* Сутки и месяцы по UTC, не по местному времени клиента.
* Август 2026 не анализировался: история кончается 30 августа, а 22–23 августа одномоментно
  закончилось ~28 тыс. подписок канала `partner_telecom` — это отдельный вопрос.